In [1]:
#!/usr/bin/env python3
"""
diagnostic_photometry_errors.py
Identifica problemas fundamentales en el pipeline de fotometría
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
import os

def analyze_identical_errors(catalog_path):
    """Analiza errores idénticos en el catálogo"""
    
    print("🔍 ANALIZANDO ERRORES IDÉNTICOS EN EL CATÁLOGO")
    print("=" * 60)
    
    df = pd.read_csv(catalog_path)
    
    # Buscar columnas de error
    error_columns = [col for col in df.columns if 'MAGERR' in col or 'FLUXERR' in col]
    
    identical_errors = {}
    
    for col in error_columns:
        # Contar valores únicos
        unique_errors = df[col].value_counts()
        
        # Buscar errores que se repiten mucho
        common_errors = unique_errors[unique_errors > 10]  # Que aparezcan más de 10 veces
        
        if len(common_errors) > 0:
            identical_errors[col] = common_errors
            print(f"\n❌ {col}:")
            for error_val, count in common_errors.head(5).items():
                print(f"   Error {error_val}: aparece {count} veces")
    
    return identical_errors

def analyze_weight_maps(field_name, filter_name):
    """Analiza los weight maps para identificar problemas"""
    
    print(f"\n🔍 ANALIZANDO WEIGHT MAPS: {field_name} {filter_name}")
    
    weight_paths = [
        f"{field_name}/{field_name}_{filter_name}.weight.fits.fz",
        f"{field_name}/{field_name}_{filter_name}.weight.fits"
    ]
    
    for weight_path in weight_paths:
        if os.path.exists(weight_path):
            try:
                with fits.open(weight_path) as hdul:
                    for hdu in hdul:
                        if hdu.data is not None:
                            weight_data = hdu.data
                            header = hdu.header
                            
                            print(f"📁 Weight map: {weight_path}")
                            print(f"   Forma: {weight_data.shape}")
                            print(f"   Tipo: {weight_data.dtype}")
                            
                            # Estadísticas del weight map
                            valid_weights = weight_data[np.isfinite(weight_data) & (weight_data > 0)]
                            
                            if len(valid_weights) > 0:
                                print(f"   Mínimo: {np.min(valid_weights):.6f}")
                                print(f"   Máximo: {np.max(valid_weights):.6f}")
                                print(f"   Mediana: {np.median(valid_weights):.6f}")
                                print(f"   Media: {np.mean(valid_weights):.6f}")
                                print(f"   Std: {np.std(valid_weights):.6f}")
                                
                                # Verificar valores constantes
                                unique_vals = np.unique(valid_weights)
                                print(f"   Valores únicos: {len(unique_vals)}")
                                if len(unique_vals) < 10:
                                    print(f"   Valores: {unique_vals}")
                                
                                # Porcentaje de ceros o valores inválidos
                                zeros = np.sum(weight_data == 0)
                                nans = np.sum(~np.isfinite(weight_data))
                                total_pixels = weight_data.size
                                
                                print(f"   Ceros: {zeros}/{total_pixels} ({zeros/total_pixels*100:.2f}%)")
                                print(f"   NaN/Inf: {nans}/{total_pixels} ({nans/total_pixels*100:.2f}%)")
                                
                            else:
                                print("   ⚠️  No hay pesos válidos")
                            
                            break
                    else:
                        print("   ⚠️  No se encontraron datos en el archivo")
            except Exception as e:
                print(f"   ❌ Error leyendo {weight_path}: {e}")
            break
    else:
        print("   ⚠️  No se encontró weight map")

def analyze_flux_distribution(catalog_path):
    """Analiza la distribución de flujos para identificar problemas"""
    
    print(f"\n📊 ANALIZANDO DISTRIBUCIÓN DE FLUJOS")
    
    df = pd.read_csv(catalog_path)
    flux_columns = [col for col in df.columns if 'FLUX_' in col and 'FLUXERR' not in col]
    
    for col in flux_columns[:5]:  # Solo primeros 5 para no saturar
        fluxes = df[col][df[col] < 1e6]  # Filtrar valores extremos
        fluxes = fluxes[np.isfinite(fluxes)]
        
        if len(fluxes) > 0:
            print(f"\n{col}:")
            print(f"   Mínimo: {np.min(fluxes):.6f}")
            print(f"   Máximo: {np.max(fluxes):.6f}")
            print(f"   Mediana: {np.median(fluxes):.6f}")
            print(f"   Ceros: {np.sum(fluxes == 0)}")
            print(f"   Negativos: {np.sum(fluxes < 0)}")

def check_background_subtraction(field_name, filter_name):
    """Verifica el proceso de resta de fondo"""
    
    print(f"\n🌌 VERIFICANDO RESTA DE FONDO: {field_name} {filter_name}")
    
    image_paths = [
        f"{field_name}/{field_name}_{filter_name}.fits.fz",
        f"{field_name}/{field_name}_{filter_name}.fits"
    ]
    
    for image_path in image_paths:
        if os.path.exists(image_path):
            try:
                with fits.open(image_path) as hdul:
                    for hdu in hdul:
                        if hdu.data is not None:
                            data = hdu.data.astype(float)
                            header = hdu.header
                            
                            print(f"📁 Imagen: {image_path}")
                            print(f"   Forma: {data.shape}")
                            print(f"   Rango de datos: {np.min(data):.3f} a {np.max(data):.3f}")
                            
                            # Estadísticas de fondo
                            from astropy.stats import sigma_clipped_stats
                            mean, median, std = sigma_clipped_stats(data, sigma=3.0)
                            
                            print(f"   Fondo (sigma-clipped):")
                            print(f"     Media: {mean:.6f}")
                            print(f"     Mediana: {median:.6f}")
                            print(f"     Std: {std:.6f}")
                            
                            # Porcentaje de píxeles negativos
                            negative_pixels = np.sum(data < 0)
                            total_pixels = data.size
                            print(f"   Píxeles negativos: {negative_pixels}/{total_pixels} ({negative_pixels/total_pixels*100:.2f}%)")
                            
                            break
            except Exception as e:
                print(f"   ❌ Error leyendo {image_path}: {e}")
            break

def main():
    """Función principal de diagnóstico"""
    
    catalog_path = "../anac_data/Results/all_fields_gc_photometry_corrected_errors_v17.csv"
    
    if not os.path.exists(catalog_path):
        print(f"❌ No se encuentra el catálogo: {catalog_path}")
        return
    
    # 1. Analizar errores idénticos
    identical_errors = analyze_identical_errors(catalog_path)
    
    # 2. Analizar weight maps para algunos campos/filtros de ejemplo
    test_fields_filters = [
        ('CenA01', 'F660'),
        ('CenA01', 'F861'), 
        ('CenA09', 'F660')
    ]
    
    for field, filt in test_fields_filters:
        analyze_weight_maps(field, filt)
    
    # 3. Analizar distribución de flujos
    analyze_flux_distribution(catalog_path)
    
    # 4. Verificar resta de fondo
    for field, filt in test_fields_filters[:2]:
        check_background_subtraction(field, filt)
    
    print("\n" + "=" * 60)
    print("🎯 RECOMENDACIONES BASADAS EN EL DIAGNÓSTICO:")
    
    if identical_errors:
        print("❌ PROBLEMA CRÍTICO: Errores idénticos detectados")
        print("   Esto indica problemas en:")
        print("   - Weight maps (valores constantes o incorrectos)")
        print("   - Propagación de errores en el código")
        print("   - Fondo sobre-restado o mal estimado")
    
    print("\n🔧 SOLUCIONES SUGERIDAS:")
    print("1. Verificar que los weight maps sean válidos y no constantes")
    print("2. Revisar la función de propagación de errores en el código")
    print("3. Comprobar la estimación del fondo (puede estar sobre-restado)")
    print("4. Considerar usar una metodología más simple sin correcciones complejas")
    print("5. Validar con fuentes brillantes conocidas")

if __name__ == "__main__":
    main()

🔍 ANALIZANDO ERRORES IDÉNTICOS EN EL CATÁLOGO

❌ FLUXERR_F378_2:
   Error inf: aparece 244 veces

❌ MAGERR_F378_2:
   Error 0.5428681023790647: aparece 1942 veces
   Error 99.0: aparece 1282 veces

❌ FLUXERR_F378_3:
   Error inf: aparece 244 veces

❌ MAGERR_F378_3:
   Error 0.5428681023790647: aparece 1896 veces
   Error 99.0: aparece 1320 veces

❌ FLUXERR_F395_2:
   Error inf: aparece 244 veces

❌ MAGERR_F395_2:
   Error 0.5428681023790647: aparece 2019 veces
   Error 99.0: aparece 1336 veces

❌ FLUXERR_F395_3:
   Error inf: aparece 244 veces

❌ MAGERR_F395_3:
   Error 0.5428681023790647: aparece 2025 veces
   Error 99.0: aparece 1322 veces

❌ FLUXERR_F410_2:
   Error inf: aparece 245 veces

❌ MAGERR_F410_2:
   Error 0.5428681023790647: aparece 1891 veces
   Error 99.0: aparece 1080 veces

❌ FLUXERR_F410_3:
   Error inf: aparece 245 veces

❌ MAGERR_F410_3:
   Error 0.5428681023790647: aparece 1826 veces
   Error 99.0: aparece 1172 veces

❌ FLUXERR_F430_2:
   Error inf: aparece 243 vec